In [9]:
import pandas as pd
import numpy as np
import os
import pickle
import json
# for content_based filtering
from sklearn.metrics.pairwise import cosine_similarity

# for collabaractive filtering

In [2]:
movieId_lookup = pd.read_csv('../dataset/processed/movieId_lookup.csv')
tmdb_movies_info = pd.read_csv('../dataset/processed/movies_df_clean.csv')

In [3]:
#  filter movies which have rated by users. 
tmdb_movies_info = tmdb_movies_info[tmdb_movies_info['movie_id'].isin(movieId_lookup['tmdb_movieId'])].reset_index(drop=True)

In [4]:
with open('../artifact/svd_model.pkl','rb') as file:
    svd_model = pickle.load(file)

In [5]:
combined_embeddings = np.load('../artifact/embeddings/final_embeddings.npy')
director_embeddings = np.load('../artifact/embeddings/director_embeddings.npy')
keyword_embeddings = np.load('../artifact/embeddings/keyword_embeddings.npy')
genre_embeddings = np.load('../artifact/embeddings/genre_embeddings.npy')
overview_embeddings = np.load('../artifact/embeddings/overview_embeddings.npy')


In [6]:

def content_based_filtering(movie_idx):
    '''
    apply content based filtering on given movie index and return the top similarities movies.
    input:
    movie_idx : index value of movieId_lookup table.

    return :
    top similarity movies list.
    '''
   
    # find the index of tmdb movie based on index value
    tmdb_movieId = movieId_lookup.iloc[movie_idx]['tmdb_movieId']
    
    # calculate movie similarity based on embedded text vector metrix.
    tmdb_index = tmdb_movies_info[tmdb_movies_info['movie_id'] == tmdb_movieId].index[0]
    similarities = cosine_similarity([combined_embeddings[tmdb_index]],combined_embeddings)[0]

    # sort index based on values then change the order into descending and give first 10 values index
    top_indicies = similarities.argsort()[::-1][:11]

    

    # for similary_tmdb_id in top_indicies:
    return tmdb_movies_info.loc[top_indicies,'title']
    

In [7]:
content_based_filtering(movieId_lookup.sample(1).index[0])

1756    10 Things I Hate About You
2067                      The DUFF
926             Something Borrowed
1994          World's Greatest Dad
1158            Notes on a Scandal
1742                      Clueless
2161            Outside Providence
1883              Cruel Intentions
1595                Boys and Girls
711                  Love Actually
548                          Hitch
Name: title, dtype: str

In [17]:
PATH_MOVIELENS = os.path.join('..','dataset','MovieLens')

movielens_movies = pd.read_csv(os.path.join(PATH_MOVIELENS,'movies.csv'))


In [22]:
def collaborative_based_filtering(user_id,movie_watched):
    '''
    apply collaborative based filtering on given movie index and return the top similarities movies.
    input:
    movie_idx : index value of movieId_lookup table.

    return :
    top similarity movies list.
    ''' 

     # find the index of movielens movie based on index value
    # get all movies which exist in dataset.
    all_movies = set(movieId_lookup['movieLens_movieId'])
    if not type(movie_watched) == set:
        movie_watched = set(movie_watched)
        
    unwatched_movies = all_movies = movie_watched

    
    predictions = []
    # calculate the prediction rating for user U for unwatched movie uM
    for movie in unwatched_movies:
        pred_rating = svd_model.predict(user_id,movie).est
        predictions.append([movie,pred_rating])
       
    
    # sort the predicted rating by highly rated movie prediction
    predictions.sort(key=lambda x : x[1],reverse=True)
    
    movie_lookup = dict(zip(movieId_lookup['movieId'],movielens_movies['title']))
    
    for movieId, rating in predictions[:10]:
        print(movie_lookup[movieId],' : ',rating)
    

    
    

In [23]:
watched_movies = set([   1,  296,  527,  590,  750,  910,  912,  920,  953, 1090, 1097,
       1193, 1200, 1207, 1208, 1213, 1214, 1230, 1259, 1270, 1282, 1293,
       1358, 1387, 1704, 1952, 1954, 2028, 2291, 2336, 2396, 2628, 2858,
       2883, 2976, 2997, 3052, 3421, 3471, 3671, 3702])

collaborative_based_filtering(5334,watched_movies)

KeyError: 'movieId'